In [1]:
# Cell 1: Setup & Muting Warnings
!pip install -q -U transformers peft accelerate bitsandbytes trl datasets

import transformers
# THE FIX: Mute all non-critical Hugging Face warnings
transformers.logging.set_verbosity_error() 
print("✅ Libraries installed and warnings muted.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 67.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.0/531.0 kB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 625.2/625.2 kB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 66.5 MB/s eta 0:00:00
✅ Libraries installed and warnings muted.


In [2]:
# Cell 2: Data & Tokenizer Prep
from datasets import load_dataset
from transformers import AutoTokenizer

DATASET_PATH = "/kaggle/input/datasets/armanhalavi/paired-training-data/paired_training_data.jsonl" # Verify your dataset name
MODEL_ID = "NousResearch/Meta-Llama-3-8B-Instruct"

print("Loading your synthetic dataset...")
dataset = load_dataset("json", data_files=DATASET_PATH, split="train")

print("Loading Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

def format_chat_template(example):
    # THE FIX: tokenize=False safely outputs raw strings to bypass the tensor bug
    example["text"] = tokenizer.apply_chat_template(example["messages"], tokenize=False)
    return example

print("Applying Chat Template...")
dataset = dataset.map(format_chat_template)
print("✅ Data ready.")

Loading your synthetic dataset...


Generating train split: 0 examples [00:00, ? examples/s]

Loading Tokenizer...


config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

Applying Chat Template...


Map:   0%|          | 0/2887 [00:00<?, ? examples/s]

✅ Data ready.


In [3]:
# Cell 3: The Heavy Model Loader (4-Bit)
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_ID = "NousResearch/Meta-Llama-3-8B-Instruct"

print("Loading Base Model in 4-bit...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16, # Phase 3 Hardware Fix
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto"
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

print("Attaching LoRA Adapters...")
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)
model = get_peft_model(model, peft_config)
print("✅ Model and Adapters loaded.")

Loading Base Model in 4-bit...


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

Attaching LoRA Adapters...
✅ Model and Adapters loaded.


In [4]:
# Cell 4: The Checkpointed Training Loop
from trl import SFTTrainer, SFTConfig

OUTPUT_DIR = "/kaggle/working/thesis_instruct_adapter_final"

# THE FIX: We now use SFTConfig instead of TrainingArguments
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    optim="paged_adamw_32bit",
    logging_steps=10,
    save_strategy="steps", 
    save_steps=50,         
    learning_rate=2e-4,
    num_train_epochs=1, 
    fp16=False,            
    bf16=False,            
    max_grad_norm=0.3,
    warmup_ratio=0.03,
    lr_scheduler_type="constant",
    report_to="none",
    
    # THE FIX: These arguments were moved inside the config
    dataset_text_field="text",
    max_length=1024,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=training_args,
    processing_class=tokenizer, 
)

print("\n🚀 Starting the Fine-Tuning Process...")
trainer.train()

print(f"\nSaving the final adapter to {OUTPUT_DIR}...")
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("✅ Training complete! The adapter is ready for inference.")

Tokenizing train dataset:   0%|          | 0/2887 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/2887 [00:00<?, ? examples/s]


🚀 Starting the Fine-Tuning Process...
{'loss': '1.72', 'grad_norm': '0.1299', 'learning_rate': '0.0002', 'entropy': '1.515', 'num_tokens': '2.979e+04', 'mean_token_accuracy': '0.6557', 'epoch': '0.0277'}
{'loss': '1.381', 'grad_norm': '0.1572', 'learning_rate': '0.0002', 'entropy': '1.396', 'num_tokens': '6.154e+04', 'mean_token_accuracy': '0.7055', 'epoch': '0.0554'}
{'loss': '1.302', 'grad_norm': '0.1152', 'learning_rate': '0.0002', 'entropy': '1.307', 'num_tokens': '9.204e+04', 'mean_token_accuracy': '0.7167', 'epoch': '0.0831'}
{'loss': '1.313', 'grad_norm': '0.1201', 'learning_rate': '0.0002', 'entropy': '1.308', 'num_tokens': '1.245e+05', 'mean_token_accuracy': '0.7075', 'epoch': '0.1108'}
{'loss': '1.303', 'grad_norm': '0.1846', 'learning_rate': '0.0002', 'entropy': '1.297', 'num_tokens': '1.556e+05', 'mean_token_accuracy': '0.7138', 'epoch': '0.1385'}
{'loss': '1.279', 'grad_norm': '0.1123', 'learning_rate': '0.0002', 'entropy': '1.278', 'num_tokens': '1.864e+05', 'mean_token_

/usr/local/lib/python3.12/dist-packages/peft/utils/other.py:1394: UserWarning: Unable to fetch remote file due to the following error The read operation timed out - silently ignoring the lookup for the file config.json in NousResearch/Meta-Llama-3-8B-Instruct.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:295: UserWarning: Could not find a config file in NousResearch/Meta-Llama-3-8B-Instruct - will assume that the vocabulary was not modified.
  warnings.warn(


{'loss': '1.195', 'grad_norm': '0.1533', 'learning_rate': '0.0002', 'entropy': '1.221', 'num_tokens': '1.127e+06', 'mean_token_accuracy': '0.7372', 'epoch': '0.9972'}
{'train_runtime': '8525', 'train_samples_per_second': '0.339', 'train_steps_per_second': '0.042', 'train_loss': '1.228', 'entropy': '1.146', 'num_tokens': '1.13e+06', 'mean_token_accuracy': '0.7303', 'epoch': '1'}

Saving the final adapter to /kaggle/working/thesis_instruct_adapter_final...
✅ Training complete! The adapter is ready for inference.
